# 02 資料處理入門：從 CSV 到分析就緒的 DataFrame

松柏護理之家退伍軍人症群聚事件，280 筆個案名冊已彙整完成。
這堂課我們把 CSV 讀進 pandas，檢查品質，建立衍生變項，做出翼區侵襲率統計表。

## 什麼是 DataFrame？

**pandas** 是 Python 最常用的資料處理套件，你可以把它想成「Python 版的 Excel」。

- **DataFrame** = 一張二維表格（行 × 列），就像 Excel 工作表
- **Series** = 一個一維欄位，就像 Excel 裡的「一行」

這堂課的所有操作都圍繞著 DataFrame 展開。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: 讀入 line list ---
# import pandas as pd → 匯入 pandas，取綽號 pd（全世界的約定）
# pd.read_csv() → 讀取 CSV 檔案，回傳一個 DataFrame
# df.shape → (列數, 欄數)，[0] 取列數、[1] 取欄數
# df.head() → 顯示前 5 筆（可加數字如 df.head(10)）

import pandas as pd

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
print(f"資料維度：{df.shape[0]} 筆 × {df.shape[1]} 欄")
df.head()

In [ ]:
# --- Step 2: 檢視資料結構 ---
# df.info() 告訴你：
#   - 每個欄位的「名稱」和「型別」
#     (int64=整數, float64=小數, object=文字, bool=布林)
#   - 每個欄位有幾個「非空值」→ 280 以下就有遺漏值
# 💡 object 型別通常代表文字，日期讀進來也是 object，需要手動轉換

df.info()

In [ ]:
# df.describe() 告訴你每個數值欄位的統計摘要：
#   count=非空值數, mean=平均, std=標準差
#   min/max=最小最大, 25%/50%/75%=四分位數
# 重點：年齡 min/max 合理嗎？有沒有 -1 或 999 這種異常值？

df.describe()

In [ ]:
# --- Step 3: 日期轉換 ---
# CSV 讀進來的日期是「文字」(object)，Python 不知道它是日期
# 必須用 pd.to_datetime() 轉成 datetime 型別，才能做排序、相減、取月份等操作
#
# pd.to_datetime(df[col]) → 文字 "2026-01-15" 變成 datetime 物件
# errors="coerce" → 遇到空白或 "N/A" 不報錯，改成 NaT (Not a Time，日期的遺漏值)
# df[col] = ... → 把轉換結果存回原本的欄位

date_cols = [
    "facility_admission_date",
    "symptom_onset_date",
    "hospitalization_date",
    "death_date",
    "notification_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# 驗證轉換結果：應該都變成 datetime64[ns]，不再是 object
df[date_cols].dtypes

In [ ]:
# --- Step 4: 建立衍生變項 ---
# 語法：df["新欄位名"] = 計算公式（跟 Excel 新增一欄的概念一樣）

# 1) 年齡組 — pd.cut() 把連續數字分組（像考試分 A/B/C/D 等級）
#    bins=[59,69,79,89,100] 是切割點（左開右閉）
#    (59,69]=60-69歲, (69,79]=70-79歲, ...
df["age_group"] = pd.cut(
    df["age"],
    bins=[59, 69, 79, 89, 100],
    labels=["60-69", "70-79", "80-89", "90+"],
)

# 2) 共病數 — .sum(axis=1) 是「橫向加總」（對每個人，加總他的 5 個共病欄位）
#    axis=0 = 往下（每欄的平均）; axis=1 = 往右（每列的加總）
comorbidity_cols = [
    "comorbidity_chf", "comorbidity_dm",
    "comorbidity_cancer", "comorbidity_copd",
    "immunosuppressed",
]
df["n_comorbidities"] = df[comorbidity_cols].sum(axis=1)

# 3) 是否感染 — 布林運算（!= "not_ill" → True/False）再轉成 0/1
#    .astype(int) 把 True→1, False→0
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# 4) 發病到住院天數 — 兩個日期相減得到時間差，.dt.days 取天數
#    .dt 是「日期存取器」：.dt.year(年) .dt.month(月) .dt.days(天數差)
df["onset_to_hosp_days"] = (
    df["hospitalization_date"] - df["symptom_onset_date"]
).dt.days

# 5) 流行病學週 — ISO 8601 標準的週次（1~53），疫調常用來做「每週統計」
df["epi_week"] = df["symptom_onset_date"].dt.isocalendar().week

# 檢視新增的欄位
df[["case_id", "age", "age_group", "n_comorbidities", "infected",
    "onset_to_hosp_days", "epi_week"]].head(10)

In [ ]:
# --- Step 5: 處理遺漏值 ---
# 遺漏值 = 資料中的「空格」，pandas 用三種符號表示：
#   NaN (Not a Number) → 數字欄位的遺漏
#   NaT (Not a Time)   → 日期欄位的遺漏
#   None               → Python 原生的「沒有值」
#
# df.isnull() → 每個格子回傳 True/False（是否為遺漏值）
# .sum()      → 對每一欄加總 True 的數量（True=1, False=0）
# missing[missing > 0] → 只顯示有遺漏值的欄位（布林篩選）
#
# ⚠️ 「結構性遺漏」vs「資料錯誤」：
#   - 未感染者沒有發病日期 → 結構性遺漏（正常，不需要填補）
#   - 感染者的年齡欄位空白 → 資料錯誤（需要回頭補登）

print("=== 各欄位遺漏值數量 ===")
missing = df.isnull().sum()
print(missing[missing > 0].to_string())

# df.loc[條件, 欄位名] → 先篩選列（條件），再取特定欄位
# .notna() → 與 .isnull() 相反，True 代表「有值」
# .sum()   → 加總有值的數量
print(f"\n未感染者有 onset 日期的數量："
      f"{df.loc[df['infected'] == 0, 'symptom_onset_date'].notna().sum()}")
print("→ 0 表示結構性遺漏沒有問題")

In [ ]:
# --- Step 6: groupby 分組統計 ---
# groupby 是 pandas 最強大的功能之一，概念就像 Excel 的「樞紐分析表」：
#   1) 分組 (Split)：把資料按 floor × wing 切成小組
#   2) 套用 (Apply)：對每組計算統計量
#   3) 合併 (Combine)：把結果拼回一張表
#
# df.groupby(["floor", "wing"]) → 按樓層和翼區分組
# .agg(                         → 對每組套用多個統計函數
#     residents=("case_id", "size"),   → 計算每組的列數（= 住民數）
#     infected=("infected", "sum"),    → 加總 infected 欄位（1+1+0+... = 感染人數）
# )
# .reset_index() → 把 groupby 產生的多層索引「攤平」回普通欄位
#
# ⚠️ 侵襲率 = 感染人數 ÷ 該區住民數（不是除以全體 280 人！）
#    分母搞錯是疫調報告最常見的錯誤之一

wing_stats = (
    df.groupby(["floor", "wing"])
    .agg(residents=("case_id", "size"), infected=("infected", "sum"))
    .reset_index()
)
wing_stats["attack_rate"] = wing_stats["infected"] / wing_stats["residents"]
wing_stats["attack_rate_pct"] = (wing_stats["attack_rate"] * 100).round(1)

print("=== 各翼區侵襲率 ===")
print(wing_stats.to_string(index=False))

In [ ]:
# --- Step 6b: 進階資料操作 ---
# 以下是 Excel 使用者轉換到 pandas 時最常用的進階技巧

# === 頻率表：value_counts() ===
# 拿到資料第一件事：看每個欄位的次數分布（像 Excel 的 COUNTIF）
print("=== 臨床嚴重度分布 ===")
print(df["clinical_severity"].value_counts())
print("\n百分比：")
print((df["clinical_severity"].value_counts(normalize=True) * 100).round(1))

# === 樞紐分析表：pivot_table() ===
# 就是 Excel 的 Pivot Table！index=列標籤, columns=欄標籤, values=值
pivot = pd.pivot_table(
    df,
    values="infected",
    index="wing",
    columns="floor",
    aggfunc="mean",
    margins=True,
    margins_name="合計",
)
print("\n=== 各翼區 × 樓層侵襲率 (%) ===")
print((pivot * 100).round(1))

# === 交叉表：crosstab() ===
# 快速建立 2×2 表（Ch03 會深入教）
print("\n=== 性別 × 感染狀態 ===")
print(pd.crosstab(df["sex"], df["infected"], margins=True))

In [ ]:
# === Method Chaining：一行寫完分析 ===
# 傳統寫法需要很多暫時變數，Method Chaining 把多步操作串成流水線
# .query() 用字串寫篩選條件（and/or/not 取代 &/|/~）
# .assign() 在鏈上直接新增欄位

# 傳統寫法 vs Method Chaining
# 傳統：
# cases = df[df["infected"] == 1]
# elderly = cases[cases["age"] >= 80]
# result = elderly.groupby("floor").size().reset_index(name="n")

# Method Chaining（一氣呵成）：
result = (
    df
    .query("infected == 1 and age >= 80")
    .groupby("floor")
    .size()
    .reset_index(name="n_elderly_cases")
    .sort_values("n_elderly_cases", ascending=False)
)
print("=== 80歲以上感染者，依樓層 ===")
print(result.to_string(index=False))

# 更複雜的 chaining：一次算侵襲率和致死率
summary = (
    df
    .assign(dead=(df["outcome"] == "dead").astype(int))
    .groupby("floor")
    .agg(
        n=("case_id", "size"),
        infected=("infected", "sum"),
        deaths=("dead", "sum"),
    )
    .assign(
        attack_rate=lambda d: (d["infected"] / d["n"] * 100).round(1),
        cfr=lambda d: (d["deaths"] / d["infected"] * 100).round(1),
    )
)
print("\n=== 各樓層侵襲率 + 致死率 ===")
print(summary)

In [ ]:
# === merge：合併資料表（VLOOKUP 等價物）===
# 疫調中常需要合併不同來源的資料（個案名冊 + 檢驗結果）
# pd.merge(左表, 右表, on=共同欄位, how=合併方式)
# how="left" → 保留左表所有列（最常用，不遺漏個案）

# 模擬一份檢驗結果表
lab_df = df[df["lab_confirmed"] == True][["case_id"]].head(10).copy()
lab_df["ct_value"] = [25.3, 28.1, 22.5, 31.0, 24.8, 27.2, 23.1, 29.5, 26.0, 30.2]

# 合併到前 20 位住民
sample = df[["case_id", "age", "sex", "infected"]].head(20)
merged = pd.merge(sample, lab_df, on="case_id", how="left")
print("=== merge 結果（前 20 位住民 + 檢驗 ct_value）===")
print(merged.to_string(index=False))
print(f"\n合併前：{len(sample)} 列 → 合併後：{len(merged)} 列（how='left' 不遺漏）")

# === .str 字串操作 + drop_duplicates ===
# .str accessor 讓你對整欄文字做操作（不用寫迴圈）
print("\n=== 翼區欄位統一大寫 ===")
print(df["wing"].str.upper().value_counts())

# drop_duplicates：去除重複（疫調中重複通報很常見）
print(f"\n去重前：{len(df)} 筆")
df_unique = df.drop_duplicates(subset="case_id", keep="first")
print(f"去重後：{len(df_unique)} 筆")

# nlargest：快速找出前 N 名
print("\n=== 侵襲率最高的前 3 個翼區 ===")
print(wing_stats.nlargest(3, "attack_rate_pct")[["floor", "wing", "attack_rate_pct"]].to_string(index=False))

## 小結

這堂課你完成了 line list 資料處理的完整流程：

**基礎六步驟**
1. **讀入** CSV → `pd.read_csv()`
2. **檢視** 結構 → `df.info()`, `df.describe()`
3. **轉換** 日期 → `pd.to_datetime()`
4. **衍生** 新變項 → `pd.cut()`, `.sum(axis=1)`, `.dt.days`
5. **確認** 遺漏值 → 結構性遺漏 vs 資料錯誤
6. **統計** 分組指標 → `groupby().agg()`

**進階操作（Step 6b）**
- **頻率表** → `value_counts()`, `pd.crosstab()`
- **樞紐分析** → `pd.pivot_table()`（Excel Pivot Table 等價物）
- **Method Chaining** → `.query().groupby().assign()` 流水線
- **合併資料表** → `pd.merge()`（VLOOKUP 等價物）
- **文字清理** → `.str.upper()`, `.str.contains()`, `drop_duplicates()`

下一堂課我們用這份清理好的資料來畫圖——流行曲線、年齡分布、翼區比較、熱力圖、互動圖。

> 📄 **pandas 速查表**：[Pandas Cheat Sheet (PDF)](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf)